# 03 — CV evaluation: held-out mAP, per-class report, ONNX parity

**Run on Kaggle** after `02_yolov8_finetune_cardd_vehide.ipynb`.

Attach as inputs:
1. The **combined dataset** (from notebook 01's saved output)
2. The **training output** — either attach notebook 02's output directly (Add Input → Your Work → notebook 02), or a dataset you made from `best.pt`/`best.onnx`

Produces `/kaggle/working/cv_eval_report.json` — the single source of truth for every CV number
quoted in `docs/ARCHITECTURE.md`, the README, and the Phase 11 deck. **These are the honest,
held-out numbers — never quote training-set metrics.**

In [ ]:
%pip install -q ultralytics onnxruntime
import glob, json, yaml
from pathlib import Path

UNIFIED = ["dent","scratch","crack","glass_shatter","lamp_broken","tire_flat","punctured","missing_part"]

# locate combined dataset
data_yamls = glob.glob("/kaggle/input/**/data.yaml", recursive=True)
assert data_yamls, "Attach the combined dataset (notebook 01 output)."
root = Path(data_yamls[0]).parent
print("dataset root:", root)

# locate trained weights
pts = sorted(glob.glob("/kaggle/input/**/best.pt", recursive=True))
assert pts, "Attach notebook 02's output (contains best.pt)."
BEST_PT = pts[0]
onnxs = sorted(glob.glob("/kaggle/input/**/best.onnx", recursive=True))
print("weights:", BEST_PT, "| onnx:", onnxs[:1])

### Held-out split
Notebook 01 produced `train`/`val`. The val split was used for early stopping in training, so for a
strictly-held-out number we carve a **test subset** the model never influenced: a deterministic
(seeded) 50% half of val, fixed by filename hash — same files every run, no leakage from re-runs.

In [ ]:
import hashlib, shutil

val_imgs = sorted(glob.glob(str(root/"images/val/*")))
test_imgs = [p for p in val_imgs if int(hashlib.md5(Path(p).name.encode()).hexdigest(), 16) % 2 == 0]
print(f"val total: {len(val_imgs)} -> held-out test half: {len(test_imgs)}")

# materialise the test split in working dir (input is read-only)
TEST = Path("/kaggle/working/testset")
for sub in ["images/test", "labels/test"]:
    (TEST/sub).mkdir(parents=True, exist_ok=True)
for img in test_imgs:
    shutil.copy(img, TEST/"images/test"/Path(img).name)
    lbl = root/"labels/val"/(Path(img).stem + ".txt")
    if lbl.exists():
        shutil.copy(lbl, TEST/"labels/test"/lbl.name)

TEST_YAML = "/kaggle/working/test_data.yaml"
Path(TEST_YAML).write_text(yaml.safe_dump({
    "path": str(TEST), "train": "images/test", "val": "images/test",
    "nc": len(UNIFIED), "names": UNIFIED,
}))
print("test yaml written")

### Evaluate on the held-out test half

In [ ]:
from ultralytics import YOLO
m = YOLO(BEST_PT)
r = m.val(data=TEST_YAML, split="val", plots=True)

per_class = {}
for i, cid in enumerate(r.box.ap_class_index):
    per_class[UNIFIED[int(cid)]] = {
        "precision": round(float(r.box.p[i]), 4),
        "recall":    round(float(r.box.r[i]), 4),
        "mAP50":     round(float(r.box.ap50[i]), 4),
        "mAP50_95":  round(float(r.box.ap[i]), 4),
    }

report = {
    "eval_split": f"held-out deterministic half of val ({len(test_imgs)} images)",
    "overall": {
        "mAP50":    round(float(r.box.map50), 4),
        "mAP50_95": round(float(r.box.map), 4),
        "precision_mean": round(float(r.box.mp), 4),
        "recall_mean":    round(float(r.box.mr), 4),
    },
    "per_class": per_class,
}
print(json.dumps(report, indent=2))

### ONNX parity check
The HF Space serves `best.onnx` via onnxruntime, not `best.pt` — verify the exported graph produces equivalent detections before shipping it.

In [ ]:
import numpy as np, onnxruntime as ort, cv2

if onnxs:
    sess = ort.InferenceSession(onnxs[0], providers=["CPUExecutionProvider"])
    inp = sess.get_inputs()[0]
    print("onnx input:", inp.name, inp.shape)

    def letterbox(img, size=640):
        h, w = img.shape[:2]
        s = size / max(h, w)
        nh, nw = int(round(h*s)), int(round(w*s))
        canvas = np.full((size, size, 3), 114, np.uint8)
        canvas[:nh, :nw] = cv2.resize(img, (nw, nh))
        return canvas

    sample = test_imgs[:5]
    agree = []
    for p in sample:
        img = cv2.imread(p)
        x = letterbox(img).astype(np.float32).transpose(2,0,1)[None] / 255.0
        onnx_out = sess.run(None, {inp.name: x})[0]           # (1, 4+nc, 8400)
        onnx_n = int((onnx_out[0, 4:, :].max(axis=0) > 0.25).sum())
        pt_n = len(m.predict(p, conf=0.25, verbose=False)[0].boxes)
        agree.append((Path(p).name, pt_n, onnx_n))
    for name, a, b in agree:
        print(f"{name}: pt={a} raw-onnx-candidates={b}")
    report["onnx_parity"] = {"checked_images": len(sample),
                             "note": "raw ONNX candidate count >= NMS'd pt count is expected; Space applies its own NMS"}
else:
    report["onnx_parity"] = {"skipped": "best.onnx not attached"}

In [ ]:
with open("/kaggle/working/cv_eval_report.json", "w") as f:
    json.dump(report, f, indent=2)
print("wrote /kaggle/working/cv_eval_report.json")
print("\nCopy this file into the repo at eval/cv_eval_report.json and quote ONLY these numbers in docs/ARCHITECTURE.md.")

### Reading the numbers honestly
- **mAP50** — mean average precision at IoU 0.5: the headline detection-quality number.
- **mAP50-95** — stricter localisation average; always lower, report it anyway.
- Class imbalance (e.g. `tire_flat` is rare) will show as weaker per-class recall — name that in the
  Limitations slide rather than hiding it. A modest, real number beats an inflated one under audit.